# GenAI Intensive 4일차 예제 노트북 · 검색 품질 개선 · RAG 평가체계 · LLM Wiki 비교용

강의안의 4일차 예제 코드가 모두 이 노트북 하나에 있습니다. 셀 제목의 번호(예제 1-1 의 1. 등)는 강의안 코드 장 제목의 번호와 같습니다.
위에서부터 순서대로 실행합니다. 파트 1 은 검색 품질 개선, 파트 2 는 RAG 평가체계입니다.
파트 3 「LLM Wiki 와 온톨로지」의 예제 3-1 · 3-2 와 실습 미션 3 은 노트북이 아니라 Claude 앱 Code 탭에서 `day04-wiki/` 폴더로 합니다. 이 노트북 맨 아래에는 예제 3-1 에서 위키에 물은 두 질문을 RAG 로 똑같이 물어 보는 비교용 절 하나만 있습니다.
실습 미션 1 · 2 는 따로 받은 `day04-lab.ipynb` 에서 합니다.

**처음 한 번만 할 일 · 코랩**
1. [파일] → [Drive에 사본 저장] 을 선택하고, 새 탭에 열린 사본에서 진행합니다.
2. 왼쪽 열쇠 아이콘(보안 비밀)에 2일차에 넣은 `ANTHROPIC_API_KEY` 가 있는지 보고, 그 줄의 「노트북 액세스」를 켭니다. 없으면 「새 보안 비밀 추가」에서 이름 `ANTHROPIC_API_KEY`, 값에 받은 키를 붙여 넣습니다.
3. 아래 「환경 준비」 셀 (1) 재료와 라이브러리, (2) 키와 연결 확인, (3) 3일차 색인, (4) 평가셋을 위에서부터 차례로 실행합니다(Shift + Enter). 확인할 출력은 둘입니다.
- 환경 준비 (2) 셀: `연결 확인:` 뒤에 수도 이름이 나옵니다.
- 환경 준비 (4) 셀: `평가셋 33 문항 · 검색 채점 대상 31 문항` 이 나옵니다.

둘 다 나오면 준비가 끝난 것입니다.

환경 준비 (3) 을 처음 실행하면 임베딩 모델 `intfloat/multilingual-e5-small` 을, 예제 1-4 (1) 을 처음 실행하면 리랭커 모델을 내려받습니다. 런타임이 끊기면 환경 준비 셀부터 다시 실행합니다.

### Claude Code 에 물어보는 법

셀의 코드가 궁금하면 셀을 통째로 복사해 PC 의 Claude Code 에 붙이고, 아래 셋 가운데 하나를 덧붙여 묻습니다. Claude Code 가 알려 준 코드는 그 셀 바로 아래 새 셀에 붙여 실행합니다. 그 코드가 `chunks` · `search` 처럼 노트북에 있는 이름을 다시 만들었다면, 원래 셀을 한 번 더 실행하고 다음 예제로 넘어갑니다.

1. 「이 셀에서 ○○ 줄이 하는 일을 한 문장으로 알려 줘」
2. 「○○ 값을 △△ 로 바꿔 실행하면 결과가 어떻게 달라지는지 알려 줘」
3. 「이 오류가 왜 났고 어떻게 고치는지 알려 줘」(오류 메시지 전문을 함께 붙입니다)

셀마다 맨 끝 주석 `# Claude Code 에 물어보기:` 에 그 셀에 맞는 질문을 하나씩 적어 두었습니다.

## 환경 준비

**환경 준비 (1) 재료를 받고 라이브러리를 설치한다**
- 학습 사이트에서 재료 zip(문서 폴더 `corpus/`, 문서 목록, 평가셋, 채점 프롬프트 파일)을 받아 현재 폴더에 풉니다. `corpus/` 폴더가 이미 있으면 받지 않습니다.
- `%pip` 줄은 오늘 쓰는 라이브러리 세 개를 설치합니다. `anthropic` 은 Claude API, `rank-bm25` 는 키워드 검색, `sentence-transformers` 는 임베딩 모델과 리랭커에 씁니다.
- 설치 중에 「restart the kernel」 안내가 나와도 런타임을 다시 시작하지 않고 다음 셀로 갑니다.
- **출력에서 볼 것**: 마지막 줄이 `문서 파일 253 개` 이면 재료가 제대로 풀린 것입니다. 253개는 문서 252건에 옛 판본 한 건(SOP-ETCH-012 v1.2)을 더한 수입니다.

In [ ]:
# 환경 준비 (1) 재료 받기와 설치 · 노트북을 열면 한 번 실행합니다
# 코랩에서는 재료 zip 을 학습 사이트에서 받아 이 자리에 풉니다.
# 이미 corpus/ 폴더가 있으면 받지 않습니다.
import os, zipfile, urllib.request
URL = 'https://lecture-web-five.vercel.app/materials/day04v2-examples.zip'
if not os.path.exists('corpus'):
    urllib.request.urlretrieve(URL, 'day04v2-examples.zip')
    zipfile.ZipFile('day04v2-examples.zip').extractall('.')
# -q 는 설치 로그를 줄여서 보여 줍니다
%pip -q install anthropic rank-bm25 sentence-transformers
print('문서 파일', len(os.listdir('corpus')), '개')

# Claude Code 에 물어보기: corpus 폴더에서 SOP-ETCH-012 파일이 몇 개인지 집계하는 코드 짜 줘

**환경 준비 (2) API 키를 넣고 Claude 연결을 확인한다**
- 코랩 「보안 비밀」에서 `ANTHROPIC_API_KEY` 를 읽고, 없으면 `getpass` 입력창으로 받습니다.
- `text_of(resp)` 는 응답 블록 가운데 text 블록만 이어 붙여 문자열로 반환합니다.
- `llm(prompt, max_tokens)` 는 프롬프트 하나를 Claude 에 보내고 답 문자열을 반환합니다. 같은 프롬프트의 답은 `_MEMO` 사전에 저장해 두고, 다시 들어오면 요청을 보내지 않고 저장된 답을 가져와 씁니다. 오늘 쓰는 LLM 호출은 모두 이 함수를 거칩니다.
- **출력에서 볼 것**: `연결 확인:` 뒤에 수도 이름이 나오면 키와 모델이 연결된 것입니다. 오류가 나면 보안 비밀 이름과 「노트북 액세스」 설정을 확인합니다.

In [ ]:
# 환경 준비 (2) API 키와 모델 · 2일차와 같은 방식입니다
# 코랩 왼쪽 열쇠 모양 「보안 비밀」에 ANTHROPIC_API_KEY 를 넣고 「노트북 액세스」를 켭니다.
import os
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY').strip()
# 코랩이 아니거나 보안 비밀이 없으면 넘어갑니다
except Exception:
    pass
if not os.environ.get('ANTHROPIC_API_KEY'):
    try:
        import getpass
        os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY: ')
    except Exception:
        print('[안내] 키가 없습니다 · 보안 비밀 이름 ANTHROPIC_API_KEY 를 확인합니다')

import anthropic
client = anthropic.Anthropic()
MODEL = 'claude-sonnet-5-5'                # 2일차 기본 모델과 같습니다

def text_of(resp):
    """응답의 text 블록만 이어 붙인 문자열을 돌려줍니다. thinking 블록은 건너뜁니다."""
    return ''.join(block.text for block in resp.content if block.type == 'text').strip()

# 뒤 예제 셀을 다시 실행해도 같은 요청은 두 번 보내지 않습니다
_MEMO = {}
# 2000 은 오늘 예제 답 길이의 기본값 · 답이 잘리면 호출할 때 늘립니다
def llm(prompt, max_tokens=2000):
    """프롬프트 하나를 Claude 에 보내고 답 문자열을 돌려줍니다."""
    if prompt in _MEMO:
        return _MEMO[prompt]
    resp = client.messages.create(
        model=MODEL, max_tokens=max_tokens,
        # 추론 노력을 낮춰 응답 시간을 줄입니다
        output_config={'effort': 'low'},
        messages=[{'role': 'user', 'content': prompt}])
    reply = text_of(resp)
    if not reply:                          # 빈 답은 저장하지 않고 이유를 보여 줍니다
        return f'[빈 응답] stop_reason={resp.stop_reason} · max_tokens 를 늘립니다'
    _MEMO[prompt] = reply
    return reply

print('연결 확인:', llm('한 단어로만 답하세요. 대한민국의 수도는?'))

# Claude Code 에 물어보기: _MEMO 에 쌓인 프롬프트 수와 총 글자 수를 출력하는 코드 짜 줘

**환경 준비 (3) 3일차 색인을 다시 세운다**
3일차 예제 3-1 (a) 조립 셀에 예제 3-3 (f) 옛 판본 제외와 예제 3-3 (h) 문서 제목 붙이기를 더한 규칙으로 다시 세웁니다. 셀 안의 이름은 아래와 같습니다.
- `model`: 문장을 384차원 벡터로 바꾸는 임베딩 모델(multilingual-e5-small)입니다. 질문 앞에는 `query: `, 문서 앞에는 `passage: ` 를 붙여 넣습니다.
- `load_chunks()`: `corpus/` 의 .md 파일을 `## ` 절 단위로 잘라 청크 목록을 반환합니다. 청크마다 `corpus_manifest.csv` 에서 읽은 문서번호·판본·유형·날짜와 절 제목, 본문(1200자까지)을 담습니다.
- `chunks`: 옛 판본 SOP-ETCH-012 v1.2 를 뺀 청크 목록입니다. `TITLE`: (문서번호, 판본)마다 문서 제목을 담은 사전입니다. `E`: 청크 앞에 「문서 제목 · 절 제목」을 붙여 계산한 임베딩 행렬입니다.
- `search(q, k, exclude_types)`: `exclude_types` 에 적은 문서 유형을 빼고, 질문 벡터와 청크 벡터의 내적(정규화했으므로 코사인 유사도)을 구해 점수가 높은 청크 k개를 `Hit` 목록으로 반환합니다.
- `format_ctx(hits)`: 검색된 청크마다 `[문서번호_판본] (절 제목)` 머리를 붙여 발췌 문자열 하나로 엮습니다.
- `ask(q, k, exclude_types, template)`: `search` → `format_ctx` → `TEMPLATE` 에 끼우기 → `llm` 호출을 이어서 실행하고 (답, 검색 결과)를 반환합니다.
- `DOCT`: 문서번호를 넣으면 그 문서의 청크 목록을 가져오는 사전입니다. 문서번호가 `doc_id` 일 때 `DOCT[doc_id][0].section` 이 문서 제목입니다.
- **출력에서 볼 것**: 처음 실행하면 모델을 내려받고 754청크를 임베딩하느라 시간이 걸립니다. 앞에 나오는 진행 표시(Loading weights …)는 넘기고, 마지막 줄이 `청크 754 · 문서 252 · 벡터 (754, 384)` 이면 3일차와 같은 색인입니다.

In [ ]:
# 환경 준비 (3) 3일차 마지막 색인(예제 3-1 (a) + 3-3 (f) · (h))을 다시 세웁니다
#   · corpus/*.md 를 '## ' 절 단위로 자릅니다 (머리 청크는 앞 8줄, 본문은 1200자까지)
#   · 3일차 3-3 (f) 처럼 SOP-ETCH-012 의 옛 판본 v1.2 를 뺍니다
#   · 3일차 3-3 (h) 처럼 청크 앞에 「문서 제목 · 절 제목」 을 붙여 임베딩합니다
import re, csv, json, time, unicodedata
import numpy as np
from collections import namedtuple, defaultdict, Counter
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('intfloat/multilingual-e5-small')
META = {row['file']: row for row in
        csv.DictReader(open('corpus_manifest.csv', encoding='utf-8-sig'))}
Chunk = namedtuple('Chunk', 'doc_id version type date section text')

def load_chunks(folder='corpus'):
    """corpus/ 의 .md 파일을 '## ' 절 단위로 자른 Chunk 목록을 돌려줍니다."""
    loaded = []
    for raw_name in sorted(os.listdir(folder)):
        file_name = unicodedata.normalize('NFC', raw_name)  # 목록표 조회용 이름
        if not file_name.endswith('.md'):
            continue
        meta_row = META[file_name]
        text = open(f'{folder}/{raw_name}', encoding='utf-8').read()
        # '## ' 줄 앞에서 자르고 첫 조각은 head 에 둡니다
        head, *sections = re.split(r'\n(?=## )', text)
        for section in [head] + sections:
            section = section.strip()
            if sections and section == head.strip():
                section = '\n'.join(section.splitlines()[:8])
            section_title = section.splitlines()[0].lstrip('# ').strip()
            loaded.append(Chunk(meta_row['doc_id'], meta_row['version'],
                                meta_row['type'], meta_row['date'],
                                section_title, section[:1200]))
    return loaded

chunks = [chunk for chunk in load_chunks()
          if not (chunk.doc_id == 'SOP-ETCH-012' and chunk.version == 'v1.2')]
TITLE = {}
for chunk in chunks:                       # 첫 청크의 절 제목이 문서 제목입니다
    TITLE.setdefault((chunk.doc_id, chunk.version), chunk.section)
passages = []
for chunk in chunks:
    title = TITLE[chunk.doc_id, chunk.version]
    passages.append(f'passage: {title} · {chunk.section}\n{chunk.text}')
# 64 는 코랩 기본 메모리에서 한 번에 처리하는 청크 수입니다
E = model.encode(passages, normalize_embeddings=True, batch_size=64)

Hit = namedtuple('Hit', 'score doc_id version type date section text')
def search(q, k=4, exclude_types=()):
    """질문과 가장 비슷한 청크 k개를 점수 높은 순 Hit 목록으로 돌려줍니다."""
    question_vector = model.encode(['query: ' + q], normalize_embeddings=True)[0]
    candidates = [i for i, chunk in enumerate(chunks)
                  if chunk.type not in exclude_types]
    scores = E[candidates] @ question_vector
    # 오름차순 정렬을 뒤집어 점수 높은 순 k개를 고릅니다
    top_positions = scores.argsort()[::-1][:k]
    return [Hit(round(float(scores[position]), 3), *chunks[candidates[position]])
            for position in top_positions]

def format_ctx(hits):
    """검색 결과를 [문서번호_판본] (절 제목) 머리를 붙인 발췌 문자열로 돌려줍니다."""
    return '\n\n'.join(f'[{hit.doc_id}_{hit.version}] ({hit.section})\n{hit.text}'
                       for hit in hits)

TEMPLATE = '''너는 SL-FAB 공정기술 담당자다.
아래 사내 문서 발췌만 근거로 답한다.

[발췌]
{ctx}

규칙: 문장마다 근거 [문서번호]를 붙인다.
발췌에 없으면 '사내 문서에서 근거를 찾지 못했습니다' 라고만 답한다.

질문: {q}'''

def ask(q, k=4, exclude_types=(), template=None):
    """검색 → 발췌 → 템플릿 → llm 호출을 이어 (답, 검색 결과) 를 돌려줍니다."""
    hits = search(q, k, exclude_types)
    return llm((template or TEMPLATE).format(ctx=format_ctx(hits), q=q)), hits

DOCT = defaultdict(list)                   # 문서번호 → 그 문서의 청크들
for chunk in chunks:
    DOCT[chunk.doc_id].append(chunk)
print('청크', len(chunks), '· 문서', len(DOCT), '· 벡터', E.shape)

# Claude Code 에 물어보기: load_chunks 를 '### ' 소절로도 자르면 청크가 몇 개가 되는지 집계해 줘

**환경 준비 (4) 평가셋을 불러온다**
- `EV_ID`·`EV_AUTO`: 평가셋 두 벌입니다. 문항마다 질문 `q`, 정답 문서번호 `gold`, 문서 유형 `type` 이 들어 있습니다. `EV_ID` 는 질문에 날짜·호기 같은 식별자를 넣은 평가셋, `EV_AUTO` 는 문서만 주고 자동 생성한 평가셋입니다.
- **출력에서 볼 것**: `EV_ID` 기준으로 `평가셋 33 문항 · 검색 채점 대상 31 문항` 이 나오면 됩니다. 거절해야 하는 1문항(`gold` 가 NONE)과 정답 문서를 정하지 않은 1문항(`gold` 가 None)은 검색 채점에서 빠집니다.

In [ ]:
# 환경 준비 (4) 오늘 쓰는 평가셋 두 벌을 불러옵니다 · 검색기와 채점 함수는 예제 1-1 에서 만듭니다
# EV_ID 는 질문에 식별자를 넣은 평가셋, EV_AUTO 는 문서만 주고 자동 생성한 평가셋입니다
EV_ID   = json.load(open('evalset_id.json', encoding='utf-8'))
EV_AUTO = json.load(open('evalset_auto.json', encoding='utf-8'))

# 정답 문서가 없는 문항은 채점에서 뺍니다
scored = sum(item['gold'] not in (None, 'NONE') for item in EV_ID)
print('평가셋', len(EV_ID), '문항 · 검색 채점 대상', scored, '문항')

# Claude Code 에 물어보기: EV_ID 와 EV_AUTO 를 type 별 문항 수 표로 나란히 보여 줘

## 파트 1 · 검색 품질 개선

예제 1-1 ~ 1-5 입니다. 예제 1-1 에서 3일차 벡터 검색기를 평가셋으로 다시 측정한 뒤, 예제 1-2 키워드 검색 · 예제 1-3 하이브리드 검색 · 예제 1-4 리랭킹을 차례로 더해 같은 평가셋으로 비교하고, 예제 1-5 에서 남은 실패를 유형으로 나눕니다. 이 파트는 Claude API 를 호출하지 않습니다.

## 예제 1-1. 상위 검색 결과를 직접 열어 봅니다

**1. 질문 하나의 상위 검색 결과를 연다**: 3일차에 순위를 끌어올린 질문입니다

(심화 · 수업에서 넘어갈 수 있음) 이 셀을 넘어가도 뒤의 기본 셀은 이 셀이 만든 이름 `q` 를 쓰지 않으므로 그대로 실행됩니다.

**이 셀이 하는 일**: 3일차에 순위를 끌어올린 질문 하나를 `search(q, k=4)` 에 넣고 상위 4개 청크의 점수·문서번호·절 제목을 출력합니다.

**출력에서 볼 것**: 정답 문서 QIR-2026-0712 의 청크 세 개가 2~4위에 있고 1위는 SOP-ETCH-007 의 「배경」 절입니다. 맨 앞 숫자는 질문과 청크의 코사인 유사도이며 1에 가까울수록 뜻이 비슷합니다.

In [ ]:
q = 'ETCH-04 호기 ESC 온도 드리프트 원인과 재발 방지 조치는?'
for hit in search(q, k=4):
    print(hit.score, hit.doc_id, hit.section[:20])

# Claude Code 에 물어보기: k=10 이면 QIR-2026-0712 청크가 몇 개인지 집계해 줘

**2. 검색이 틀린 문항 하나를 연다**: 정답 문서가 몇 위에 있고, 1위에는 무엇이 올라왔는지 봅니다

**이 셀이 하는 일**: 먼저 3일차 벡터 검색기를 문서 순위로 돌려주는 함수 셋을 만듭니다.
- `vec_rank(q, n)`: `search()` 와 같은 벡터 점수로 청크 번호 상위 n개를 반환합니다.
- `to_docs(idx, k)`: 청크 순위를 문서 순위로 바꿉니다. 같은 문서의 청크가 여러 번 나오면 처음 나온 자리 하나만 집계합니다.
- `vector_search(q)`: 위 두 함수를 이어 문서번호 10개를 순위대로 반환합니다.

그다음 식별자 평가셋 `EV_ID` 에서 정답이 MAN-013 인 문항을 가져와 `vector_search` 상위 3건과 각 문서 제목을 출력합니다. `next(item for item in EV_ID if ...)` 는 조건에 맞는 첫 항목 하나를 가져옵니다.

**출력에서 볼 것**: 정답 MAN-013 은 2위이고 1위는 MAN-023 입니다. 1~3위 모두 드라이 펌프 장비 매뉴얼로, 제목 꼴이 같다는 점을 확인합니다.

In [ ]:
# 3일차 벡터 검색기를 문서 순위로 돌려주게 만듭니다 · 예제 1-1 부터 계속 씁니다
# 50 은 문서 10건으로 접기에 넉넉한 청크 수입니다
def vec_rank(q, n=50):
    """벡터 검색 순위. search() 와 같은 계산이고 청크 번호 상위 n개만 돌려줍니다."""
    question_vector = model.encode(['query: ' + q], normalize_embeddings=True)[0]
    scores = E @ question_vector
    # 점수가 같으면 청크 번호 순을 지킵니다
    return list(np.argsort(-scores, kind='stable')[:n])

def to_docs(idx, k=10):
    """청크 순위 idx 를 문서 순위로 접은 문서번호 목록(k개까지)을 돌려줍니다."""
    # 같은 문서의 청크는 처음 나온 자리 하나만 집계합니다
    doc_ids = []
    for i in idx:
        if chunks[i].doc_id not in doc_ids:
            doc_ids.append(chunks[i].doc_id)
        if len(doc_ids) >= k:
            break
    return doc_ids

def vector_search(q):
    """3일차에 만든 벡터 검색기. 문서 10건을 순위대로 돌려줍니다."""
    return to_docs(vec_rank(q), 10)

e = next(item for item in EV_ID if item['gold'] == 'MAN-013')
print('질문 ', e['q'])
print('정답 ', e['gold'])
# enumerate(..., 1) 은 순위를 1부터 매깁니다
for rank, doc_id in enumerate(vector_search(e['q'])[:3], 1):
    print(rank, '위', doc_id, DOCT[doc_id][0].section[:24])

# Claude Code 에 물어보기: to_docs 가 문서마다 가장 높은 청크 점수도 함께 돌려주게 고쳐 줘

**3. 1위 문서와 정답 문서의 본문을 나란히 본다**

**이 셀이 하는 일**: 예제 1-1 (2) 의 1위 문서를 `top` 에 담고, `top` 과 정답 문서에서 마지막 청크의 본문을 줄마다 출력해 나란히 비교합니다.

**출력에서 볼 것**: 두 문서는 문장이 같고 확인 주기(주 단위·월 단위)와 시운전 횟수(1회·4회)만 다릅니다. 서식이 같고 숫자만 다른 문서는 벡터 점수가 비슷하게 나와 벡터 검색이 잘 구분하지 못합니다.

In [ ]:
# 예제 1-1 (3) 1위 문서와 정답 문서의 마지막 청크 본문을 나란히 출력합니다
top = vector_search(e['q'])[0]
for doc_id in [top, e['gold']]:
    print(doc_id, DOCT[doc_id][0].section[:22])
    # 마지막 청크의 본문 줄(첫 줄 절 제목은 뺌)
    for line in DOCT[doc_id][-1].text.splitlines()[1:]:
        print('   ', line[:44])

# Claude Code 에 물어보기: MAN-023 과 MAN-013 의 첫 청크(문서 머리)도 나란히 보여 줘

**4. 문항 31개를 하나씩 볼 수 없으니 평가셋으로 측정한다**: hit@k 와 MRR

**이 셀이 하는 일**: 먼저 채점 함수 둘을 만듭니다.
- `metrics(search_fn, evset)`: 문항마다 검색기를 실행해 정답 문서의 순위를 찾고 hit@1 · hit@3 · MRR 을 계산합니다. 선택 인수 `gold_sets` 를 따로 주면 정답 문서 여러 건 가운데 하나만 나와도 맞은 것으로 판정합니다.
- `show(name, m)`: `metrics` 결과를 한 줄로 출력합니다.

그다음 `metrics` 로 벡터 검색기를 채점 대상 31문항 전체에 실행해 지표를 계산합니다. `metrics` 에는 함수 이름만 넣고 괄호를 붙이지 않습니다.
- hit@k: 정답 문서가 상위 k건 안에 든 문항의 비율(%)입니다.
- MRR(Mean Reciprocal Rank): 정답 순위의 역수(1위 1, 2위 0.5, 3위 0.33)를 문항마다 구해 평균한 값입니다. 1에 가까울수록 정답이 위에 있고, 10위 안에 없으면 0으로 계산합니다.

**출력에서 볼 것**: hit@1 67.7 은 31문항 중 21문항이 1위로 맞았다는 뜻입니다. hit@3 93.5 는 31문항 중 29문항이 3위 안에 들었다는 뜻입니다. 이 줄이 뒤 예제와 비교할 기준 줄입니다.

In [ ]:
# 검색기 하나를 평가셋으로 채점하는 함수 · 예제 1-2 부터 모든 검색기를 이 함수로 측정합니다
def metrics(search_fn, evset, gold_sets=None):
    """검색기 하나를 평가셋 하나로 측정해 hit@1 · hit@3 · MRR 을 돌려줍니다."""
    hit1_count = hit3_count = scored = 0
    reciprocal_rank_sum = 0.0
    for e in evset:
        # 정답 문서가 없는 문항은 검색 채점에서 뺍니다
        if e['gold'] in (None, 'NONE'):
            continue
        gold = set(gold_sets.get(e['q'], [e['gold']])) if gold_sets else {e['gold']}
        ranked = search_fn(e['q'])
        scored += 1
        # 정답이 처음 나온 위치(0부터), 없으면 None
        position = next((i for i, doc_id in enumerate(ranked) if doc_id in gold), None)
        if position is not None:
            reciprocal_rank_sum += 1 / (position + 1)
            hit1_count += position < 1       # 비교 결과 True 는 1 로 더해집니다
            hit3_count += position < 3
    return dict(hit1=round(hit1_count / scored * 100, 1),
                hit3=round(hit3_count / scored * 100, 1),
                mrr=round(reciprocal_rank_sum / scored, 3), n=scored)

def show(name, result):
    """검색기 이름과 metrics 결과를 받아 hit@1 · hit@3 · MRR 을 한 줄로 출력합니다."""
    print(f"{name:6s} hit@1 {result['hit1']:5.1f}   hit@3 {result['hit3']:5.1f}   MRR {result['mrr']:.3f}")

show('벡터', metrics(vector_search, EV_ID))

# Claude Code 에 물어보기: metrics 에 hit@5 를 더하고 show 에도 같이 출력되게 고쳐 줘

**5. 1위를 놓친 문항만 모아 본다**

**이 셀이 하는 일**: `misses` 함수를 새로 만들어, 검색기 1위가 정답이 아닌 문항마다 (정답 문서, 정답 순위, 1위 문서)를 모읍니다. 10위 안에 정답이 없으면 순위는 None 입니다.

**출력에서 볼 것**: 10문항이 나오고 8문항은 정답이 2~3위, 나머지는 4위 · 5위입니다. 줄마다 정답과 1위의 문서번호를 비교해, 매뉴얼 · 안전규정 · 공지처럼 같은 종류의 문서인 문항이 몇 개인지 집계해 둡니다.

In [ ]:
# 1위를 놓친 문항을 모으는 함수 · 벡터 검색기로 실행해 문항별 정답 순위를 출력합니다
def misses(search_fn, evset):
    """1위를 놓친 문항마다 (정답 문서, 정답 순위, 1위 문서) 를 모아 돌려줍니다."""
    missed = []
    for e in evset:
        if e['gold'] in (None, 'NONE'):
            continue
        ranked = search_fn(e['q'])
        if ranked[0] != e['gold']:
            # 위치(0부터)에 1을 더해 순위로 바꿉니다 · 10위 안에 없으면 None
            rank = ranked.index(e['gold']) + 1 if e['gold'] in ranked else None
            missed.append((e['gold'], rank, ranked[0]))
    return missed

for gold, rank, top in misses(vector_search, EV_ID):
    print(f'{gold:13s} 정답 {rank}위 · 1위 {top}')

# Claude Code 에 물어보기: misses 결과를 정답 순위별 문항 수로 Counter 로 집계해 줘

## 예제 1-2. 키워드 검색과 벡터 검색 결과를 나란히 비교합니다

**1. BM25 색인을 만든다**: 같은 754청크를 문서 제목을 붙여 단어 단위로 넣습니다

**이 셀이 하는 일**: 같은 754청크의 본문 앞에 문서 제목을 붙여(환경 준비 (3) 의 벡터 색인과 같은 조건) 단어로 자른 뒤 BM25 키워드 색인 `bm25` 를 만들고, 청크 순위를 돌려주는 `bm_rank` 와 문서 10건을 돌려주는 `keyword_search` 를 정의합니다.
- BM25: 질문 단어가 청크에 몇 번 나오는지, 그 단어가 전체 청크에서 얼마나 드문지, 청크가 얼마나 긴지로 점수를 매기는 키워드 검색 방식입니다. 글자가 정확히 같아야 점수가 오르므로 날짜·호기명·문서번호에 강합니다.
- `tok`: 소문자로 바꾸고 기호를 공백으로 바꾼 뒤 단어로 자릅니다. 세 글자 이상 한글 단어는 두 글자 조각도 더해, 조사가 붙은 단어(「파티클이」)와 조각이 겹치게 합니다.

**출력에서 볼 것**: 「파티클」이 「파티」「티클」로도 들어가는지 확인합니다.

In [ ]:
# 754청크로 BM25 키워드 색인을 만들고 bm_rank · keyword_search 를 정의합니다
from rank_bm25 import BM25Okapi

def tok(text):
    """글 text 를 BM25 에 넣을 단어 조각 목록으로 돌려줍니다."""
    # 글자·숫자·한글이 아닌 기호를 공백으로 바꿉니다
    text = re.sub(r'[^\w가-힣]+', ' ', text.lower())
    tokens = []
    for word in text.split():
        tokens.append(word)
        # 조사가 붙어도 겹치도록 두 글자씩 자른 조각도 넣습니다
        if len(word) > 2 and re.match(r'^[가-힣]+$', word):
            tokens += [word[i:i + 2] for i in range(len(word) - 1)]
    return tokens

bm25 = BM25Okapi([tok(TITLE[chunk.doc_id, chunk.version] + ' ' + chunk.text)
                  for chunk in chunks])

# 50: vec_rank 와 같은 후보 수 · 예제 1-3 에서 두 순위를 RRF 로 합칠 때 씁니다
def bm_rank(q, n=50):
    """BM25 점수가 큰 청크 번호 n개를 돌려줍니다."""
    return list(np.argsort(-bm25.get_scores(tok(q)), kind='stable')[:n])

def keyword_search(q):
    """키워드 검색기. 문서 10건을 순위대로 돌려줍니다."""
    return to_docs(bm_rank(q), 10)

print(tok('ETCH-04 호기 파티클 급증'))

# Claude Code 에 물어보기: tok 에서 두 글자 조각을 빼면 '파티클 급증' 토큰이 어떻게 바뀌는지 보여 줘

**2. 같은 질문을 두 검색기에 넣고 상위 3건을 나란히 비교한다**

(심화 · 수업에서 넘어갈 수 있음) 넘어가도 뒤의 기본 셀은 이 셀이 만든 이름을 쓰지 않아 그대로 실행됩니다.

**이 셀이 하는 일**: 정답이 NOTI-005 인 문항의 질문을 벡터·키워드 두 검색기에 넣고 상위 3건을 순위마다 한 줄씩 나란히 출력합니다. `zip(vector_top3, keyword_top3)` 는 두 목록에서 같은 순위의 항목을 짝지어 가져옵니다.

**출력에서 볼 것**: 둘째 줄(1위)을 봅니다. 벡터는 같은 제목의 다른 공지 NOTI-017 을 1위로 올리고, 키워드는 날짜 단어가 겹친 정답 NOTI-005 를 1위로 올립니다.

In [ ]:
# 정답이 NOTI-005 인 문항을 벡터·키워드 검색기에 넣고 상위 3건을 나란히 출력합니다
e = next(x for x in EV_ID if x['gold'] == 'NOTI-005')
# 한 줄에 맞추려 질문은 앞 46자만 출력합니다
print('질문', e['q'][:46], '· 정답', e['gold'])
vector_top3 = vector_search(e['q'])[:3]
keyword_top3 = keyword_search(e['q'])[:3]
# 같은 순위의 두 결과를 짝지어 가져옵니다
for rank, (vector_doc, keyword_doc) in enumerate(zip(vector_top3, keyword_top3), 1):
    print(f'{rank}위  벡터 {vector_doc:13s} 키워드 {keyword_doc}')

# Claude Code 에 물어보기: NOTI-005 와 NOTI-017 의 제목·날짜에서 겹치는 단어를 뽑아 줘

**3. 식별자가 들어간 평가셋으로 두 검색기를 측정한다**

**이 셀이 하는 일**: 식별자 평가셋 `EV_ID` 로 두 검색기의 hit@1·hit@3·MRR 을 측정합니다. 함수 이름만 바꿔 넣으면 같은 `metrics` 로 어느 검색기든 측정할 수 있습니다.

**출력에서 볼 것**: 첫 줄 벡터의 hit@1 이 67.7, 둘째 줄 키워드가 74.2 입니다. 질문에 날짜·호기 같은 식별자가 있으면 글자가 정확히 겹치는 키워드 검색이 유리합니다.

In [ ]:
# 식별자 평가셋 EV_ID 로 벡터·키워드 두 검색기를 측정해 출력합니다
show('벡터', metrics(vector_search, EV_ID))
show('키워드', metrics(keyword_search, EV_ID))

# Claude Code 에 물어보기: EV_ID 에서 벡터는 1위인데 키워드는 놓친 문항만 질문을 보여 줘

**4. 식별자 없이 자동 생성한 평가셋으로도 측정한다**

(심화 · 수업에서 넘어갈 수 있음) 넘어가도 뒤의 기본 셀은 이 셀이 만든 이름을 쓰지 않아 그대로 실행됩니다.

**이 셀이 하는 일**: 식별자를 넣지 않고 문서만 주고 생성한 평가셋 `EV_AUTO` 로 같은 두 검색기를 측정합니다.

**출력에서 볼 것**: hit@1 이 벡터 58.1, 키워드 61.3 으로 두 검색기의 차이가 식별자 평가셋(예제 1-2 (3))보다 줄고, hit@3 은 두 검색기 모두 87.1 입니다. 질문에서 식별자가 빠지면 키워드 검색의 이점이 작아지는지 확인합니다.

In [ ]:
# 자동 생성 평가셋 EV_AUTO 로 벡터·키워드 두 검색기를 측정해 출력합니다
show('벡터', metrics(vector_search, EV_AUTO))
show('키워드', metrics(keyword_search, EV_AUTO))

# Claude Code 에 물어보기: EV_AUTO 질문 중 날짜·호기 같은 숫자가 든 문항이 몇 개인지 집계해 줘

## 예제 1-3. 두 검색 결과를 합칩니다: 하이브리드 검색

**1. 한 문서가 두 목록에서 각각 몇 위인지 본다**

(수업에서는 셀만 실행하고 설명은 넘어갈 수 있음) 이 셀이 만든 `e` · `vr` · `kr` 은 예제 1-3 (2) 와 (3) 에서 쓰므로 이 셀은 꼭 실행합니다.

**이 셀이 하는 일**: 정답이 MTG-2025-W06 인 회의록 질문으로 벡터 순위 `vr` 과 키워드 순위 `kr`(각각 문서 10건)을 만들고, 정답 문서와 벡터 상위 2건이 두 목록에서 각각 몇 위인지 출력합니다. 목록에 없으면 `-` 로 표시합니다.

**출력에서 볼 것**: 정답 W06 은 벡터 3위·키워드 2위로 두 목록 모두 위쪽에 있습니다. 벡터 1위 W11 은 키워드 10위 안에 없습니다.

In [ ]:
# 회의록 질문 하나로 벡터·키워드 문서 순위를 만들고 정답과 벡터 상위 2건의 순위를 출력합니다
e = next(x for x in EV_ID if x['gold'] == 'MTG-2025-W06')
vr = to_docs(vec_rank(e['q']), 10)
kr = to_docs(bm_rank(e['q']), 10)
# 정답과 벡터 상위 2건(정답과 겹치면 뺌)
for doc_id in [e['gold']] + [x for x in vr[:2] if x != e['gold']]:
    # 목록에 있으면 순위(1부터), 없으면 '-'
    vector_rank = vr.index(doc_id) + 1 if doc_id in vr else '-'
    keyword_rank = kr.index(doc_id) + 1 if doc_id in kr else '-'
    print(f'{doc_id:13s} 벡터 {vector_rank}위 · 키워드 {keyword_rank}위')

# Claude Code 에 물어보기: vr 과 kr 두 목록에 모두 든 문서가 몇 건인지 집계하는 코드 짜 줘

**2. 등수를 점수로 바꿔 더한다**: RRF 는 등수 r 을 1/(60+r) 로 바꿔 더합니다

(심화 · 수업에서 넘어갈 수 있음) 넘어가도 뒤의 기본 셀은 이 셀이 만든 이름을 쓰지 않아 그대로 실행됩니다.

**이 셀이 하는 일**: 예제 1-3 (1) 에서 만든 `vr` · `kr` 을 그대로 써서 RRF(Reciprocal Rank Fusion)를 직접 계산합니다. RRF 는 순위 목록 여러 개를 합치는 방법으로, 목록마다 등수 r 을 1/(60+r) 점으로 바꿔 더합니다. 벡터 점수와 BM25 점수는 척도가 달라 그대로 더할 수 없으므로 등수만 씁니다. 60 은 1위와 2위의 점수 차가 지나치게 벌어지지 않게 하는 상수입니다.

**출력에서 볼 것**: 두 목록 모두 위에 있는 W06 이 0.0320 으로 가장 크고, 벡터 1위였지만 키워드 목록에 없는 W11 은 0.0164 로 가장 작습니다.

In [ ]:
# 정답과 벡터 상위 2건의 RRF 점수를 vr · kr 문서 순위로 직접 계산해 출력합니다
for doc_id in [e['gold']] + [x for x in vr[:2] if x != e['gold']]:
    # 들어 있는 목록마다 1/(60+등수)를 더합니다 · 60 은 상위 등수끼리 점수 차가 지나치게 벌어지지 않게 하는 RRF 상수
    rrf_score = sum(1 / (60 + ranked_list.index(doc_id) + 1)
                    for ranked_list in (vr, kr) if doc_id in ranked_list)
    print(f'{doc_id:10s} RRF {rrf_score:.4f}')

# Claude Code 에 물어보기: 60 을 1·10·100 으로 바꿔 W06 과 W11 의 RRF 점수를 표로 비교해 줘

**3. RRF 함수로 하이브리드 검색기를 만든다**

**이 셀이 하는 일**: 예제 1-3 (2) 의 계산을 `rrf(lists, k=60)` 함수로 만들고, 벡터·키워드의 청크 순위(각 50개)를 합친 뒤 문서 순위로 접는 `hybrid_search` 를 정의합니다. 예제 1-3 (2) 는 문서 순위로 계산했고, 이 함수는 청크 순위로 합칩니다.

**출력에서 볼 것**: 출력 줄의 하이브리드 1위가 정답 MTG-2025-W06 인지 확인합니다. 예제 1-3 (1) 에서 벡터 1위였던 W11 은 1위에서 내려갑니다.

In [ ]:
# 순위 목록을 합치는 rrf 와 벡터·키워드를 합친 hybrid_search 를 정의하고 1위를 확인합니다
# k=60: 상위 등수끼리 점수 차가 지나치게 벌어지지 않게 하는 RRF 상수
def rrf(lists, k=60):
    """순위 목록 여러 개를 받아 RRF 점수가 큰 순서로 청크 번호를 돌려줍니다."""
    score_by_chunk = defaultdict(float)
    for ranked_list in lists:
        for position, chunk_index in enumerate(ranked_list):
            # position 은 0부터라 1을 더해 등수로 맞춥니다
            score_by_chunk[chunk_index] += 1 / (k + position + 1)
    ranked = sorted(score_by_chunk.items(), key=lambda item: -item[1])
    return [chunk_index for chunk_index, _ in ranked]

def hybrid_search(q):
    """질문을 받아 벡터·키워드 순위를 RRF 로 합친 문서 10건을 돌려줍니다."""
    return to_docs(rrf([vec_rank(q), bm_rank(q)]), 10)

print('하이브리드 1위', hybrid_search(e['q'])[0], '· 정답', e['gold'])

# Claude Code 에 물어보기: hybrid_search 가 상위 3건의 RRF 점수도 함께 돌려주게 고쳐 줘

**4. 세 검색기를 같은 평가셋으로 측정한다**

(심화 · 수업에서 넘어갈 수 있음) 넘어가도 뒤의 기본 셀은 이 셀이 만든 이름을 쓰지 않아 그대로 실행됩니다.

**이 셀이 하는 일**: (이름, 검색기 함수) 짝 목록을 돌며 세 검색기를 같은 식별자 평가셋 `EV_ID` 로 측정합니다.

**출력에서 볼 것**: 셋째 줄 하이브리드의 hit@1 은 키워드와 같은 74.2 이고, MRR 은 키워드 0.847 에서 하이브리드 0.852 로 조금 오릅니다. hit@1 이 같은데 MRR 이 오르면 1위를 놓친 문항의 정답이 조금 더 위로 올라왔다는 뜻입니다.

In [ ]:
for name, search_fn in [('벡터', vector_search), ('키워드', keyword_search),
                        ('하이브리드', hybrid_search)]:
    show(name, metrics(search_fn, EV_ID))

# Claude Code 에 물어보기: 세 검색기 결과를 MRR 순 표로 정리해 줘

## 예제 1-4. 검색 결과 순서를 다시 매깁니다: 리랭킹 전후 비교

**1. 리랭커(크로스 인코더)를 불러온다**: 첫 실행에서 모델을 내려받습니다

**이 셀이 하는 일**: 리랭커로 쓸 크로스 인코더 모델 `rr` 을 불러옵니다.
- 크로스 인코더: 질문과 청크를 한 입력으로 붙여 넣고 관련도 점수 하나를 계산하는 모델입니다. 벡터 검색은 질문과 청크를 따로 벡터로 바꿔 비교하므로 빠르고, 크로스 인코더는 둘을 함께 계산하므로 더 정확하지만 느립니다. 그래서 앞 단계 검색이 고른 후보에만 씁니다.
- `max_length=512` 는 질문과 청크 한 쌍을 512토큰까지만 넣는다는 뜻입니다.

**출력에서 볼 것**: 모델을 불러오는 진행 표시 뒤에 `리랭커 준비 완료` 가 나오면 됩니다. 첫 실행에서는 모델 파일을 내려받느라 시간이 걸립니다.

In [ ]:
# 리랭커로 쓸 크로스 인코더 모델 rr 을 불러옵니다(첫 실행에서 내려받음)
from sentence_transformers import CrossEncoder
# 512: 질문+청크 한 쌍을 넣는 최대 토큰 수
rr = CrossEncoder('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1', max_length=512)
print('리랭커 준비 완료')

# Claude Code 에 물어보기: rr.predict 에 질문·청크 한 쌍을 넣어 점수 범위가 어떤지 보여 줘

**2. 하이브리드 상위 청크 30개를 리랭킹한다**

**이 셀이 하는 일**: `rerank_rank` 는 하이브리드 상위 청크 30개를 후보로 뽑아 (질문, 문서 제목 + 청크 앞 400자) 쌍을 만들고, `rr.predict` 로 쌍마다 관련도 점수를 계산해 높은 순으로 다시 정렬합니다. 리랭킹은 후보 30개의 순서만 바꾸므로 후보 밖의 청크는 올라오지 않습니다. `rerank_search` 는 그 순서를 `to_docs` 로 문서 10건으로 접습니다. 출력의 「전」은 `hybrid_search`, 「후」는 `rerank_search` 의 상위 3건입니다.

**출력에서 볼 것**: 「전」 줄 둘째에 있던 정답 QIR-2025-0116 이 「후」 줄 첫째로 올라옵니다. 하이브리드 2위가 리랭킹 뒤 1위가 된 것입니다.

In [ ]:
# 하이브리드 상위 청크를 리랭커로 다시 세우는 rerank_rank · rerank_search 를 정의하고 전후 상위 3건을 출력합니다
# top=30: 리랭커가 점수를 매길 후보 수 · 많을수록 느려집니다
def rerank_rank(q, top=30):
    """질문을 받아 후보 top 개를 리랭커 점수 순으로 세운 청크 번호를 돌려줍니다."""
    candidates = rrf([vec_rank(q), bm_rank(q)])[:top]
    # 리랭커에 넣을 [질문, 제목 + 본문 앞 400자] 쌍 · 400자는 max_length=512 토큰 안에 들어가게 자른 값
    pairs = []
    for chunk_index in candidates:
        chunk = chunks[chunk_index]
        pairs.append([q, TITLE[chunk.doc_id, chunk.version] + ' ' + chunk.text[:400]])
    scores = rr.predict(pairs, show_progress_bar=False)
    return [candidates[j] for j in np.argsort(-scores, kind='stable')]

def rerank_search(q):
    """질문을 받아 리랭킹한 문서 10건을 돌려줍니다."""
    return to_docs(rerank_rank(q), 10)

e = next(x for x in EV_AUTO if x['gold'] == 'QIR-2025-0116')
print('정답', e['gold'])
print('전 ', hybrid_search(e['q'])[:3])
print('후 ', rerank_search(e['q'])[:3])

# Claude Code 에 물어보기: rerank_rank 에서 rr.predict 상위 3개 점수와 청크 절 제목을 출력해 줘

**3. 두 평가셋에서 리랭킹 전후를 측정한다**

**이 셀이 하는 일**: 식별자·자동 생성 두 평가셋에서 하이브리드(전)와 리랭킹(후)의 지표를 나란히 측정합니다.

**출력에서 볼 것**: 전·후 두 줄씩 짝지어 봅니다. hit@1 은 두 평가셋 모두 오릅니다(식별자 74.2 → 80.6, 자동생성 58.1 → 67.7). hit@3 은 두 평가셋 모두 내립니다(96.8 → 93.5, 87.1 → 80.6). MRR 은 두 평가셋 모두 오릅니다(0.852 → 0.876, 0.735 → 0.782). 지표 하나만 보고 판단하지 않고 세 지표를 함께 봅니다.

In [ ]:
for name, evset in [('식별자', EV_ID), ('자동생성', EV_AUTO)]:
    show(name + ' 전', metrics(hybrid_search, evset))
    show(name + ' 후', metrics(rerank_search, evset))

# Claude Code 에 물어보기: 리랭킹 뒤 hit@3 이 내린 문항을 EV_ID 에서 찾아 줘

**4. 문항별로 오른 것과 내린 것을 집계한다**: 평균 지표에서는 안 보이는 문항별 변화를 봅니다

(심화 · 수업에서 넘어갈 수 있음) 넘어가도 뒤의 기본 셀은 이 셀이 만든 이름을 쓰지 않아 그대로 실행됩니다.

**이 셀이 하는 일**: `rank_of` 로 문항마다 리랭킹 전후의 정답 순위를 구하고, 순위가 오른 문항·내린 문항·같은 문항 수를 집계합니다. 10위 안에 정답이 없으면 순위를 99로 둡니다.

**출력에서 볼 것**: 식별자 평가셋은 오른 문항 5·내린 문항 3, 자동 생성 평가셋은 오른 문항 8·내린 문항 4 입니다. 평균이 올라도 순위가 내린 문항이 있으므로 문항별로 확인합니다.

In [ ]:
# 문항마다 리랭킹 전후의 정답 순위를 구해 오른·내린·같은 문항 수를 집계합니다
def rank_of(search_fn, e):
    """검색기와 문항을 받아 정답 문서의 순위를 돌려줍니다. 10위 안에 없으면 99 입니다."""
    ranked = search_fn(e['q'])
    # 99: 10위 안 어떤 순위보다 큰 값
    return ranked.index(e['gold']) + 1 if e['gold'] in ranked else 99

for name, evset in [('식별자', EV_ID), ('자동생성', EV_AUTO)]:
    # 문항마다 (전 순위, 후 순위) 짝을 만듭니다
    rank_pairs = [(rank_of(hybrid_search, e), rank_of(rerank_search, e))
                  for e in evset if e['gold'] not in (None, 'NONE')]
    # 후 순위 숫자가 작아지면 오른 문항입니다
    up = sum(after < before for before, after in rank_pairs)
    down = sum(after > before for before, after in rank_pairs)
    same = len(rank_pairs) - up - down
    print(name, '· 순위가 오른 문항', up, '· 내린 문항', down, '· 같은 문항', same)

# Claude Code 에 물어보기: 순위가 내린 문항의 전·후 순위를 정답 문서번호와 함께 보여 줘

**5. 걸린 시간을 같이 측정한다**

(심화 · 수업에서 넘어갈 수 있음) 넘어가도 뒤의 기본 셀은 이 셀이 만든 이름을 쓰지 않아 그대로 실행됩니다.

**이 셀이 하는 일**: `time.perf_counter()` 로 시각을 기록하면서 식별자 평가셋 31문항을 하이브리드 검색과 리랭킹 검색으로 차례로 실행합니다. 앞뒤 시각의 차를 문항 수로 나눈 값이 한 문항의 평균 시간이고, 1000 을 곱해 밀리초(ms)로 보여 줍니다.

**출력에서 볼 것**: 하이브리드와 리랭킹의 ms 값을 비교합니다. 리랭킹 값은 하이브리드 단계까지 포함한 한 문항 전체 시간이라, 두 값의 차이가 후보 30쌍을 크로스 인코더로 더 계산하는 데 든 시간입니다. 숫자 자체는 기기와 실행마다 다르므로 두 값의 차이를 봅니다.

In [ ]:
# 식별자 평가셋 31문항을 차례로 검색하며 시각을 기록합니다
questions = [e['q'] for e in EV_ID if e['gold'] not in (None, 'NONE')]
started = time.perf_counter()
for q in questions:
    hybrid_search(q)
hybrid_done = time.perf_counter()
for q in questions:
    rerank_search(q)
rerank_done = time.perf_counter()
# 전체 시간을 문항 수로 나눠 한 문항 평균을 밀리초(ms)로 바꿉니다
print('하이브리드', round((hybrid_done - started) / len(questions) * 1000), 'ms')
print('리랭킹    ', round((rerank_done - hybrid_done) / len(questions) * 1000), 'ms')

# Claude Code 에 물어보기: hybrid_search 안의 vec_rank 와 bm_rank 시간을 따로 ms 로 측정해 줘

## 예제 1-5. 못 찾은 문항을 열어 실패 유형으로 나눕니다

**1. 리랭킹 뒤에도 1위를 놓친 문항을 모은다**

(심화 · 수업에서 넘어갈 수 있음) 이 셀의 `M` 은 다음 (2) 셀(심화)만 씁니다. 두 셀을 함께 넘어가도 (3) 셀부터 그대로 실행됩니다.

**이 셀이 하는 일**: 예제 1-1 의 `misses` 에 리랭킹 검색기 `rerank_search` 와 식별자 평가셋 `EV_ID` 를 넣어, 리랭킹 뒤에도 1위를 놓친 문항을 `M` 에 모아 줄마다 출력합니다.

**출력에서 볼 것**: 줄 수가 6문항이고 매뉴얼(MAN) 넷, 안전규정(EHS) 둘인지 봅니다. 각 줄의 「정답 N위」가 모두 2~4위인지 봅니다.

In [ ]:
# 리랭킹 뒤에도 1위를 놓친 문항을 M 에 모아 정답 순위와 1위 문서를 출력합니다
M = misses(rerank_search, EV_ID)
for gold, rank, top in M:
    print(f'{gold:13s} 정답 {rank}위 · 1위 {top}')

# Claude Code 에 물어보기: M 의 6문항을 MAN·EHS 유형별로 묶어 정답 순위 평균을 내 줘

**2. 정답 문서와 1위 문서가 얼마나 닮았는지 측정한다**: 3일차 예제 3-3 에서 쓴 difflib 입니다

(심화 · 수업에서 넘어갈 수 있음) 넘어가도 뒤의 기본 셀은 이 셀이 만든 이름을 쓰지 않아 그대로 실행됩니다.

**이 셀이 하는 일**: `M` 의 문항마다 정답 문서와 1위 문서의 본문을 `difflib.SequenceMatcher` 로 비교해 닮은 정도를 0~1 로 출력합니다. `ratio()` 는 두 문자열에서 일치하는 부분의 비율이며 1.00 에 가까울수록 같은 글입니다. `body_of` 는 문서의 첫 청크(문서 머리)를 뺀 나머지 청크를 이어 붙이고, 나머지가 없으면 첫 청크를 씁니다.

**출력에서 볼 것**: 모두 0.98~1.00 입니다. 남은 실패는 본문이 거의 같은 문서에 밀린 경우입니다.

In [ ]:
# 문항마다 정답 문서와 1위 문서의 본문을 difflib 로 비교해 닮은 정도를 출력합니다
import difflib
# 첫 청크를 뺀 본문, 비었으면 첫 청크
body_of = lambda doc_id: '\n'.join(c.text for c in DOCT[doc_id][1:]) or DOCT[doc_id][0].text
for gold, _rank, top in M:
    similarity = difflib.SequenceMatcher(None, body_of(gold), body_of(top)).ratio()
    print(f'{gold:13s} vs {top:13s} 본문 닮은 정도 {similarity:.2f}')

# Claude Code 에 물어보기: MAN-013 과 1위 문서 본문에서 다른 부분만 difflib 로 뽑아 보여 줘

**3. 눈으로 본 실패에 유형 이름을 붙여 집계한다**

**이 셀이 하는 일**: `LABEL` 은 위 출력과 두 문서 원문을 열어 보고 손으로 붙인 실패 유형 이름입니다. 코드로 계산한 값이 아닙니다. 이것을 `Counter` 로 집계해 많은 순으로 출력합니다. `most_common()` 은 (값, 개수) 짝을 개수가 많은 순으로 반환합니다.

**출력에서 볼 것**: 「틀이 같고 값만 다른 문서」 4건, 「본문이 같은 문서」 2건입니다. 본문까지 같으면 정답이 둘인 문항이므로 평가셋을 고치고, 값만 다르면 질문의 식별자와 메타데이터로 가려야 합니다.

In [ ]:
# 손으로 붙인 실패 유형 이름을 Counter 로 집계해 많은 순으로 출력합니다
LABEL = {
    "MAN-012": "틀이 같고 값만 다른 문서",
    "MAN-014": "본문이 같은 문서",
    "MAN-024": "틀이 같고 값만 다른 문서",
    "MAN-013": "틀이 같고 값만 다른 문서",
    "EHS-013": "틀이 같고 값만 다른 문서",
    "EHS-007": "본문이 같은 문서"
}
for label, count in Counter(LABEL.values()).most_common():
    # 이 유형으로 분류한 문서번호를 이어 출력합니다
    doc_ids = ', '.join(doc_id for doc_id, kind in LABEL.items() if kind == label)
    print(f'{label:16s} {count}건 ·', doc_ids)

# Claude Code 에 물어보기: LABEL 을 EV_ID 의 type 과 묶어 유형별 실패 건수 표로 만들어 줘

## 파트 2 · RAG 평가체계

예제 2-1 ~ 2-6 입니다. 예제 2-1 · 2-2 에서 평가셋을 만들고, 2-3 · 2-4 에서 LLM 으로 답변을 채점하고, 2-5 에서 거절을 확인하고, 2-6 에서 병목이 검색인지 생성인지 가립니다. 이 파트의 예제는 Claude API 를 호출하니, 환경 준비 (2) 셀에서 `연결 확인:` 뒤에 수도 이름이 나왔는지 먼저 확인합니다. 예제 2-4 는 문항마다 생성 호출과 채점 호출이 한 번씩 나가므로 몇 분 걸립니다.

## 예제 2-1. 평가셋을 만듭니다: 질문과 정답 문서 짝

**1. 문서 한 건을 주고 Claude 에 질문을 만들게 한다**

**이 셀이 하는 일**: `Q_PROMPT` 에 문서 MAN-013 의 본문(앞 2500자)과 문서번호·유형·제목을 끼워 넣고 `llm` 을 호출해, 그 문서에만 있는 사실을 묻는 질문 한 줄을 생성합니다. 자동 생성 평가셋 `EV_AUTO` 는 같은 프롬프트로 문서마다 미리 만들어 둔 것입니다.

**출력에서 볼 것**: 질문이 문서 속 수치나 절차를 묻는지, 문서번호를 쓰지 않았는지 확인합니다. 질문은 런타임을 새로 열어 실행할 때마다 조금씩 달라집니다(같은 런타임에서 다시 실행하면 `_MEMO` 에 저장된 답이 그대로 나옵니다).

In [ ]:
# MAN-013 본문을 프롬프트에 끼워 그 문서에만 있는 사실을 묻는 질문 한 줄을 생성합니다
Q_PROMPT = """아래는 반도체 팹 사내 문서 한 건이다.

[문서 {did} · {typ} · {title}]
{body}

이 문서에만 있는 구체적인 사실(수치·절차·기준·조치)을 묻는 현장 질문 1개를 만들어라.
조건:
- 실제 현업 담당자가 검색창에 칠 법한 자연스러운 한국어 한 문장
- 문서번호({did})를 질문에 쓰지 말 것
- 답이 이 문서에만 있어야 한다
- 질문만 한 줄로 출력 (설명·번호·따옴표 금지)"""

did = 'MAN-013'
# 청크를 이어 붙여 앞 2500자만 씁니다
body = '\n'.join(c.text for c in DOCT[did])[:2500]
first = DOCT[did][0]
print(llm(Q_PROMPT.format(did=did, typ=first.type, title=first.section, body=body)))

# Claude Code 에 물어보기: Q_PROMPT 에 식별자(날짜·호기)를 넣으라는 조건 한 줄을 더해 다시 짜 줘

**2. 같은 방법으로 만든 질문 가운데 매뉴얼 문항을 사람이 읽어 본다**

(심화 · 수업에서 넘어갈 수 있음) 넘어가도 뒤의 기본 셀은 이 셀이 만든 이름을 쓰지 않아 그대로 실행됩니다.

**이 셀이 하는 일**: 자동 생성 평가셋 `EV_AUTO` 에서 유형이 「매뉴얼」인 문항만 골라 정답 문서와 질문을 출력합니다. 질문은 앞 44자만 출력하므로 뒤가 끊겨 보입니다.

**출력에서 볼 것**: 네 질문이 서로 어디가 다른지 비교합니다. 장비 이름(터보 펌프 · 드라이 펌프 · MFC)만 다르거나 그마저 같은지(MAN-014 · MAN-024 는 둘 다 MFC) 확인합니다. 「이 문서에만 있는」 조건을 걸어도 서식이 같은 문서에서는 비슷한 질문이 생성됩니다. 이 질문의 정답 문서가 정말 한 건인지는 예제 2-1 (3) 에서 확인합니다.

In [ ]:
for e in EV_AUTO:
    if e['type'] == '매뉴얼':
        print(e['gold'], e['q'][:44])

# Claude Code 에 물어보기: 매뉴얼 네 질문에 공통인 단어를 뽑아 줘

**3. 정답 문서가 정말 한 건인지 Claude 로 판정한다** · 참고: 벡터 상위 6건을 후보로 줍니다

> 넘어가지 않는 셀입니다. 셀을 실행한 뒤 맨 아래 출력 두 줄(정답 문서가 여럿인 문항 수 · 가장 많은 문항의 정답 문서 수)을 확인합니다. 이 셀이 만든 `gold_sets` 는 예제 2-2 (2) 에서, `GOLD_AUTO` 는 예제 2-2 (3) 에서 다시 씁니다.

**이 셀이 하는 일**: 문항마다 벡터 상위 6건을 후보로(정답 문서가 빠졌으면 더해서) Claude 에 보내고, 후보마다 질문에 완전한 답을 주는지 `문서번호: 예/아니오` 로 판정받습니다. `answerable` 은 「예」인 문서와 원래 정답을 합친 정답 집합을 반환하고, `gold_sets` 는 정답 문서가 있는 문항(거절 문항은 뺍니다)마다 {질문: 정답 집합} 사전을 반환합니다.
- `ThreadPoolExecutor(4)` 와 `ex.map` 은 LLM 호출을 네 개씩 동시에 보내 기다리는 시간을 줄입니다.

LLM 을 31번 호출하므로 출력이 나올 때까지 기다립니다. 같은 런타임에서 다시 실행하면 `_MEMO` 에 저장된 판정을 써서 바로 끝납니다.

**출력에서 볼 것**: 정답 문서가 여럿인 문항이 몇 개인지, 한 문항의 정답 문서가 가장 많게는 몇 건인지 봅니다. 판정을 LLM 이 하므로 개수는 실행마다 조금 달라질 수 있습니다.

In [ ]:
# 문항마다 벡터 상위 6건을 후보로 Claude 에 판정받아 정답 집합을 만들고 여럿인 문항 수를 출력합니다
from concurrent.futures import ThreadPoolExecutor

ANS_PROMPT = """질문: {q}

아래 후보 문서들이 이 질문에 완전한 답을 줄 수 있는지 각각 판정하라.

{cands}

각 줄에 `문서번호: 예` 또는 `문서번호: 아니오` 만 출력하라. 설명 금지."""

def answerable(e, cands):
    """문항과 후보 문서번호 목록을 받아 「예」로 판정된 문서와 원래 정답을 합친 문서번호 목록을 돌려줍니다."""
    # 후보에 정답 문서가 없으면 끝에 더합니다
    cands = cands + ([e['gold']] if e['gold'] not in cands else [])
    # 후보마다 앞 700자만 보냅니다
    doc_text = lambda doc_id: '\n'.join(c.text for c in DOCT[doc_id])[:700]
    cands_text = '\n\n'.join(f'[{doc_id}]\n' + doc_text(doc_id) for doc_id in cands)
    # 1000 은 응답 최대 토큰 수
    report = llm(ANS_PROMPT.format(q=e['q'], cands=cands_text), 1000)
    # '예' 로 판정된 줄의 문서번호만 모읍니다
    yes = set()
    for line in report.splitlines():
        doc_id, _, verdict = line.partition(':')
        if verdict.strip().startswith('예'):
            yes.add(doc_id.strip().strip('[]`'))
    return sorted((yes & set(cands)) | {e['gold']})

def gold_sets(evset):
    """평가셋을 받아 문항마다 {질문: 정답 집합} 사전을 돌려줍니다."""
    targets = [e for e in evset if e['gold'] not in (None, 'NONE')]
    cands = [vector_search(e['q'])[:6] for e in targets]   # 검색은 차례로 합니다
    # LLM 호출만 네 개씩 동시에 보냅니다
    with ThreadPoolExecutor(4) as ex:
        return dict(zip([e['q'] for e in targets], ex.map(answerable, targets, cands)))

GOLD_AUTO = gold_sets(EV_AUTO)
multi = {q: g for q, g in GOLD_AUTO.items() if len(g) > 1}
print('정답 문서가 여럿인 문항', len(multi), '/', len(GOLD_AUTO))
print('가장 많은 문항', max(len(g) for g in GOLD_AUTO.values()), '건')

# Claude Code 에 물어보기: answerable 에서 판정 줄 파싱에 실패한 건수를 같이 출력하게 고쳐 줘

**4. 정답 문서가 여럿인 문항이 어떤 유형에 몰렸는지 본다**

(심화 · 수업에서 넘어갈 수 있음) 넘어가도 뒤의 기본 셀은 이 셀이 만든 이름을 쓰지 않아 그대로 실행됩니다.

**이 셀이 하는 일**: 정답 문서가 여럿인 문항 `multi` 를 유형별로 집계하고, 정답 문서가 가장 많은 문항 하나를 질문 앞 40자와 정답 집합으로 출력합니다. `max(multi, key=...)` 는 정답 집합이 가장 긴 질문을 반환합니다.

**출력에서 볼 것**: 첫 줄의 (유형, 문항 수) 짝에서 매뉴얼·안전규정·공지처럼 서식이 같은 문서가 여러 건인 유형에 몰리는지 봅니다. 둘째 줄처럼 여러 문서에 같은 답이 실린 문항은 정답 문서를 한 건으로 두면, 다른 문서를 찾아도 오답으로 판정합니다.

In [ ]:
TYPE = {e['q']: e['type'] for e in EV_AUTO}
print(Counter(TYPE[q] for q in multi).most_common())
longest_gold_q = max(multi, key=lambda q: len(multi[q]))
print(longest_gold_q[:40], '→', multi[longest_gold_q])   # 질문은 앞 40자만 출력

# Claude Code 에 물어보기: multi 정답 문서의 제목을 TITLE 에서 보여 줘

## 예제 2-2. 질문을 고쳐 정답을 한 건으로 만들고 다시 측정합니다

**1. 식별자를 넣어 다시 만든 질문과 나란히 본다**

(심화 · 수업에서 넘어갈 수 있음) 넘어가도 뒤의 기본 셀은 이 셀이 만든 이름을 쓰지 않아 그대로 실행됩니다.

**이 셀이 하는 일**: 같은 정답 문서(NOTI-007, EHS-012)로 만든 두 질문을 「전」(`EV_AUTO`)과 「후」(`EV_ID`)로 나란히 출력합니다. 질문은 앞 40자만 출력하므로 뒤가 끊겨 보입니다.

**출력에서 볼 것**: 「후」 질문에 게시일 · 개정일 · 라인 이름 같은 식별자가 들어 있는지 확인합니다. 정답 문서가 실제로 한 건으로 좁혀졌는지는 예제 2-2 (2) 에서 봅니다.

In [ ]:
# 같은 정답 문서로 만든 전(EV_AUTO) · 후(EV_ID) 질문을 나란히 출력합니다
for gold in ['NOTI-007', 'EHS-012']:
    before = next(x for x in EV_AUTO if x['gold'] == gold)
    after = next(x for x in EV_ID if x['gold'] == gold)
    print('전', before['q'][:40])
    print('후', after['q'][:40])

# Claude Code 에 물어보기: NOTI-007 의 전·후 질문에서 새로 들어간 단어만 골라 보여 줘

**2. 같은 판정을 식별자 평가셋에도 실행한다**

**이 셀이 하는 일**: 예제 2-1 에서 만든 `gold_sets` 를 식별자 평가셋 `EV_ID` 에도 실행해 `GOLD_ID` 를 만들고, 정답 문서가 여럿인 문항 수를 집계합니다. `sum(len(g) > 1 for ...)` 은 조건이 참인 항목 수를 집계합니다. 문항마다 LLM 을 호출하므로 실행이 끝날 때까지 기다립니다.

**출력에서 볼 것**: 자동 생성 평가셋(예제 2-1 (3))보다 정답 문서가 여럿인 문항이 크게 줄었는지 봅니다. 판정을 LLM 이 하므로 개수는 실행마다 조금 달라질 수 있습니다.

In [ ]:
# 식별자 평가셋에도 gold_sets 를 실행해 정답 문서가 여럿인 문항 수를 출력합니다
GOLD_ID = gold_sets(EV_ID)
# 정답 집합이 2건 이상인 문항 수를 집계합니다
n_multi = sum(len(g) > 1 for g in GOLD_ID.values())
print('정답 문서가 여럿인 문항', n_multi, '/', len(GOLD_ID))

# Claude Code 에 물어보기: GOLD_ID 에서 여전히 정답이 여럿인 문항의 질문과 집합을 보여 줘

**3. 같은 검색기를 세 가지 평가 조건으로 측정한다**

**이 셀이 하는 일**: 벡터 검색기는 그대로 두고 평가 조건(평가셋 · 정답 기준)만 바꿔 세 번 측정합니다. 「자동·하나」는 자동 생성 질문과 정답 문서 한 건, 「식별자·하나」는 식별자 질문과 정답 문서 한 건, 「자동·집합」은 자동 생성 질문과 예제 2-1 에서 판정한 정답 집합 `GOLD_AUTO` 를 씁니다. `gold_sets=` 는 `metrics` 의 인자 이름입니다. 예제 2-2 (2) 에서 부른 함수 `gold_sets()` 와 이름만 같고 다른 것이며, 넣는 값은 예제 2-1 에서 만든 정답 집합 `GOLD_AUTO` 입니다.

**출력에서 볼 것**: 앞 두 줄은 파일에서 읽은 평가셋과 정답 문서 한 건을 쓰고 LLM 을 거치지 않으므로 hit@1 이 58.1·67.7 로 나오는지 확인합니다. 세 번째 줄은 LLM 이 판정한 정답 집합을 쓰므로 실행마다 조금 달라질 수 있습니다. 검색기가 같아도 질문(자동 · 식별자)과 정답 기준(하나 · 집합)에 따라 점수가 크게 움직이므로, 점수를 비교할 때는 평가셋과 정답 기준이 같은지 먼저 확인합니다.

In [ ]:
show('자동·하나', metrics(vector_search, EV_AUTO))
show('식별자·하나', metrics(vector_search, EV_ID))
show('자동·집합', metrics(vector_search, EV_AUTO, gold_sets=GOLD_AUTO))

# Claude Code 에 물어보기: EV_ID + GOLD_ID 조건을 네 번째 줄로 더해 줘

**4. 형식을 어긴 생성 결과를 찾는다**: 한 줄로만 출력하라고 했는데 줄이 여럿인 응답

(심화 · 수업에서 넘어갈 수 있음) 넘어가도 뒤의 기본 셀은 이 셀이 만든 이름을 쓰지 않아 그대로 실행됩니다.

**이 셀이 하는 일**: 식별자 평가셋 파일에 기록된 `raw_lines`(평가셋을 만들 때 질문 생성 응답의 줄 수)가 1보다 큰 문항을 찾습니다. `e.get('raw_lines', 1)` 은 그 키가 없으면 1 을 반환합니다.

**출력에서 볼 것**: MAN-014·MAN-024 두 건이 한 줄로만 출력하라는 조건을 어기고 2줄로 응답했습니다. 설명 한 줄 뒤에 질문이 와서 마지막 줄을 질문으로 썼습니다. 생성 결과는 형식까지 사람이 확인합니다.

In [ ]:
# 질문 생성 응답이 두 줄 이상이었던 문항을 찾아 출력합니다
for e in EV_ID:
    if e.get('raw_lines', 1) > 1:
        print(e['gold'], '응답', e['raw_lines'], '줄 → 마지막 줄을 질문으로 씀')

# Claude Code 에 물어보기: 여러 줄 응답에서 마지막 줄 대신 물음표로 끝나는 줄을 고르게 고쳐 줘

## 예제 2-3. LLM 으로 답변을 채점합니다

**1. 검색과 답변 생성을 한 함수로 잇는다**: 검색기 이름을 고를 수 있게 합니다

**이 셀이 하는 일**: 3일차 `ask` 와 같은 구조에 검색기를 고르는 `mode` 인자를 더합니다. `RANK` 사전에서 'vector'·'hybrid'·'rerank' 중 하나의 순위 함수를 가져와 상위 k개 청크를 고르고(`retrieve`), 발췌로 엮어 답을 생성합니다(`answer`). 순위 함수는 `vec_rank`(예제 1-1 (2)) · `bm_rank`(예제 1-2 (1)) · `rrf`(예제 1-3 (3)) · `rerank_rank`(예제 1-4 (2)) 를 그대로 씁니다.

**출력에서 볼 것**: 첫 줄의 발췌 문서 목록에 정답 QIR-2025-0685 가 들어 있는지, 그런데도 둘째 줄의 답(앞 120자)이 「사내 문서에서 근거를 찾지 못했습니다」(거절)로 시작하는지 봅니다. 정답 문서가 발췌에 들어와도 답이 든 절(원인·시정 조치)이 빠지면 답하지 못합니다.

In [ ]:
# 검색기를 고르는 retrieve · answer 를 정의하고 QIR-2025-0685 문항에 답을 생성합니다
# 검색기 이름 → 청크 순위를 반환하는 함수
RANK = {'vector': vec_rank, 'hybrid': lambda q: rrf([vec_rank(q), bm_rank(q)]),
        'rerank': rerank_rank}

def retrieve(q, mode='hybrid', k=4):
    """질문을 받아 mode 검색기의 상위 k개 청크를 돌려줍니다."""
    return [chunks[i] for i in RANK[mode](q)[:k]]

def answer(q, mode='hybrid', k=4):
    """질문을 받아 (답, 발췌 청크 목록)을 돌려줍니다."""
    hits = retrieve(q, mode, k)
    return llm(TEMPLATE.format(ctx=format_ctx(hits), q=q)), hits

e = next(x for x in EV_ID if x['gold'] == 'QIR-2025-0685')
ans, hits = answer(e['q'])
print([h.doc_id for h in hits])
print(ans[:120])   # 답은 앞 120자만 출력

# Claude Code 에 물어보기: k=8 로 늘리면 QIR-2025-0685 의 원인 절이 발췌에 드는지 확인해 줘

**2. 채점 프롬프트를 연다**

**이 셀이 하는 일**: 채점 프롬프트 파일 `judge_prompt.txt` 를 읽어 `JUDGE` 에 담고 마지막 8줄을 출력합니다. 파일 위쪽에는 질문·발췌·답변·정답 문서 원문을 끼울 `{q}` `{ctx}` `{ans}` `{gold}` 자리가 있습니다.

**출력에서 볼 것**: 출력 8줄에서 세 판정 항목과 출력 형식을 봅니다. 충실도는 답이 받은 발췌 안에서만 나왔는지, 근거일치는 붙인 [문서번호]가 그 내용을 담은 문서인지, 정답성은 정답 문서 원문에 비추어 사실로 맞는지를 판정합니다. 출력 형식은 예제 2-3 (3) 에서 정규식으로 판정을 뽑을 때 씁니다.

In [ ]:
# 채점 프롬프트 파일을 읽어 JUDGE 에 담고 마지막 8줄을 출력합니다
JUDGE = open('judge_prompt.txt', encoding='utf-8').read()
# 뒤에서 8줄(판정 기준·출력 형식)만 출력
for line in JUDGE.splitlines()[-8:]:
    print(line)

# Claude Code 에 물어보기: judge_prompt.txt 에서 {q} {ctx} {ans} {gold} 가 각각 어디에 끼는지 보여 줘

**3. 한 문항을 채점한다**

**이 셀이 하는 일**: `judge` 는 정답 문서 원문(앞 1800자)과 발췌(앞 3000자)를 채점 프롬프트에 끼워 `llm` 을 호출하고, 응답에서 세 항목의 판정을 정규식으로 뽑아 사전으로 반환합니다. 예제 2-3 (1) 에서 만든 문항 `e`, 답 `ans`, 발췌 `hits` 를 그대로 채점하고, 사전의 `raw`(채점 응답 원문)를 출력합니다. 이 셀은 채점 호출 한 번만 나갑니다(생성 호출은 예제 2-3 (1) 에서 이미 했습니다).

**출력에서 볼 것**: 충실도·근거일치·정답성 세 줄의 판정을 봅니다. 거절한 답은 발췌를 벗어나지 않아 충실도는 통과해도, 질문에 답하지 못했으므로 정답성은 떨어질 수 있습니다. 판정은 런타임을 새로 열어 실행할 때마다 달라질 수 있습니다(같은 런타임에서는 환경 준비 (2) 의 `llm` 이 같은 프롬프트의 답을 `_MEMO` 에 저장해 두므로 같은 판정이 그대로 나옵니다).

In [ ]:
# 채점 함수 judge 를 정의하고 예제 2-3 (1) 의 문항 하나를 채점해 보고서를 출력합니다
def judge(e, ans, hits):
    """문항 · 답 · 발췌를 받아 세 항목 판정 사전을 돌려줍니다."""
    gold = '(정답 문서 없음)'
    if e['gold'] in DOCT:
        # 정답 문서 원문은 앞 1800자만
        gold = '\n'.join(c.text for c in DOCT[e['gold']])[:1800]
    ctx = format_ctx(hits)[:3000]   # 발췌는 앞 3000자만
    # 1000 은 응답 최대 토큰 수
    report = llm(JUDGE.format(q=e['q'], ctx=ctx, ans=ans, gold=gold), 1000)
    def pick(key):
        # '충실도: 예' 꼴에서 판정 단어를 뽑습니다
        match = re.search(key + r'\s*:\s*(예|아니오|부분)', report)
        return match.group(1) if match else '?'
    return {'faith': pick('충실도'), 'cite': pick('근거일치'),
            'correct': pick('정답성'), 'raw': report}

print(judge(e, ans, hits)['raw'])

# Claude Code 에 물어보기: judge 의 pick 이 '?' 를 돌려줄 때 report 를 따로 저장하게 고쳐 줘

## 예제 2-4. 자동 평가 파이프라인: 검색기마다 채점표를 냅니다

**1. 평가셋 전체를 한 번에 채점하는 함수를 만든다**: 문항마다 호출 두 번(답변·채점)

**이 셀이 하는 일**: `eval_one` 은 문항 하나의 답을 생성하고 채점한 뒤, 정답 문서가 발췌에 있었는지(`retrieved`), 거절했는지(`refused`), 발췌 목록과 판정을 한 행(사전)으로 반환합니다. `run_eval` 은 평가셋 전체에 `eval_one` 을 실행해 행 목록을 반환하고, 함수 정의 아래에서 검색기 3종(`vector` · `hybrid` · `rerank`)의 결과를 `ROWS` 에 모읍니다.
- `refused` 는 답이 거절 문장 「사내 문서에서 근거를 찾지 못했습니다」로 시작할 때만 참입니다. 모델이 다른 말로 「없다」고 답하면 거절로 집계하지 않으니, 예제 2-4 (4) 에서 답을 직접 읽을 때 이런 경우를 확인합니다.
- 33문항 × 검색기 3종 = 99건이고, 건마다 답변·채점 호출이 두 번이라 호출은 최대 198번입니다. 검색은 차례로 하고 LLM 호출만 네 개씩 동시에 보냅니다.

**출력에서 볼 것**: `채점 99 건` 이 나오면 됩니다. 출력이 나올 때까지 몇 분 걸릴 수 있으니 셀 왼쪽 실행 표시가 도는 동안 기다립니다. 걸린 시간은 실행마다 다릅니다.

In [ ]:
# 평가셋 전체를 채점하는 eval_one · run_eval 을 정의하고 검색기 3종의 결과를 ROWS 에 모읍니다
def eval_one(e, hits, mode, k):
    """문항 하나와 발췌를 받아 답을 만들고 채점한 한 행(사전)을 돌려줍니다."""
    ans = llm(TEMPLATE.format(ctx=format_ctx(hits), q=e['q']))
    verdict = judge(e, ans, hits)
    return {'q': e['q'], 'gold': e['gold'], 'mode': mode, 'k': k,
            'retrieved': e['gold'] in {h.doc_id for h in hits},
            'refused': ans.startswith('사내 문서에서 근거를 찾지 못했습니다'),
            # 절 이름은 앞 16자만
            'excerpt': [f'{h.doc_id} · {h.section[:16]}' for h in hits],
            # **verdict 는 채점 결과 사전을 이 행에 펼침
            'ans': ans, **verdict}

def run_eval(mode, evset, k=4):
    """검색기 이름과 평가셋을 받아 문항마다 채점한 행 목록을 돌려줍니다."""
    prepared = [retrieve(e['q'], mode, k) for e in evset]   # 검색은 차례로 합니다
    # 답변 · 채점 호출만 네 개씩 동시에 보냅니다
    with ThreadPoolExecutor(4) as ex:
        # 문항과 발췌를 짝지어 eval_one 에 넘김
        return list(ex.map(lambda e, h: eval_one(e, h, mode, k), evset, prepared))

started = time.perf_counter()
ROWS = {mode: run_eval(mode, EV_ID) for mode in ['vector', 'hybrid', 'rerank']}
scored = sum(len(v) for v in ROWS.values())
print('채점', scored, '건 ·', round(time.perf_counter() - started), '초')

# Claude Code 에 물어보기: run_eval 에 진행 표시를 더해 몇 건이 끝났는지 보이게 해 줘

**2. 검색기 3종의 채점표를 나란히 놓는다**: 정답 문서가 있는 31문항 기준

**이 셀이 하는 일**: `summary` 로 검색기마다 정답 문서가 있는 31문항 기준 비율(%)을 계산합니다. 발췌적중은 정답 문서가 발췌 4건에 들어온 비율, 충실도·근거일치·정답성은 「예」 비율, 부분은 정답성이 「부분」인 비율입니다.

**출력에서 볼 것**: 검색기별 한 줄입니다. 발췌적중은 `vector` 줄 96.8, `hybrid` · `rerank` 줄 100.0 입니다. 발췌적중은 검색 결과만으로 정하므로 같은 색인이면 바뀌지 않습니다. 발췌적중이 거의 100 인데 정답성이 얼마나 낮은지 비교합니다. 단, 발췌적중은 문서번호 단위라 정답 문서의 다른 절만 들어와도 적중으로 판정합니다. 그런 문항은 예제 2-6 (3) 에서 발췌와 절 목록을 열어 봅니다. 발췌적중 밖의 칸은 LLM 판정이라 실행마다 달라집니다.

In [ ]:
def summary(rows):
    """채점 행 목록을 받아 정답 문서가 있는 문항 기준 비율(%) 사전을 돌려줍니다."""
    scored_rows = [r for r in rows if r['gold'] not in (None, 'NONE')]
    def percent_where(condition):
        return round(sum(condition(r) for r in scored_rows) / len(scored_rows) * 100, 1)
    def yes(key, value='예'):
        return percent_where(lambda r: r[key] == value)
    return {'발췌적중': percent_where(lambda r: r['retrieved']),
            '충실도': yes('faith'), '근거일치': yes('cite'),
            '정답성': yes('correct'), '부분': yes('correct', '부분')}

for mode, rows in ROWS.items():
    print(f'{mode:7s}', '  '.join(f'{k} {v:5.1f}' for k, v in summary(rows).items()))

# Claude Code 에 물어보기: summary 에 거절 비율(refused) 칸을 더해 세 검색기를 비교해 줘

**3. 점수 옆에 조건을 같이 남긴다**

(심화 · 수업에서 넘어갈 수 있음) 넘어가도 뒤의 기본 셀은 이 셀이 만든 이름을 쓰지 않아 그대로 실행됩니다.

**이 셀이 하는 일**: 날짜·모델·임베딩·청크 수·평가셋·발췌 k 를 `RUN` 사전에 적고, 채점 행 `ROWS` 와 함께 `eval_run.json` 파일에 저장합니다. 점수만 남기면 다음에 조건을 바꿔 측정한 결과와 비교할 수 없습니다.

**출력에서 볼 것**: 청크 754, 평가셋 33문항이 출력되는지 봅니다(채점표는 그중 정답 문서가 있는 31문항 기준). 날짜는 실행한 날로 바뀝니다. 코랩 왼쪽 파일 탭에 `eval_run.json` 이 생겼는지 보고, 남겨 둘 것이면 내려받습니다. 런타임 연결이 끊기면 지워집니다.

In [ ]:
# 측정 조건을 RUN 사전에 적고 채점 행 ROWS 와 함께 eval_run.json 에 저장합니다
RUN = {'날짜': time.strftime('%Y-%m-%d'), '생성·채점 모델': MODEL,
       '임베딩': 'multilingual-e5-small', '청크': len(chunks),
       # 예제 2-4 채점 때의 발췌 건수
       '평가셋': f'evalset_id {len(EV_ID)}문항', '발췌 k': 4}
print(RUN)
with open('eval_run.json', 'w', encoding='utf-8') as f:
    # ensure_ascii=False 로 한글을 그대로 저장
    json.dump({'run': RUN, 'rows': ROWS}, f, ensure_ascii=False)

# Claude Code 에 물어보기: eval_run.json 을 다시 읽어 RUN 과 rows 건수를 확인하는 코드 짜 줘

**4. 채점 결과를 사람이 몇 건 열어 본다**: 채점도 LLM 이 하므로 틀릴 수 있습니다

(심화 · 수업에서 넘어갈 수 있음) 넘어가도 뒤의 기본 셀은 이 셀이 만든 이름을 쓰지 않아 그대로 실행됩니다.

**이 셀이 하는 일**: 리랭킹 결과에서 정답 문서가 있는 문항 가운데 정답성이 「예」가 아닌 문항을 골라 앞 두 건의 정답 문서번호 · 정답성, 질문, 답 앞부분을 출력합니다. 채점도 LLM 이 하므로 몇 건은 사람이 답을 다시 봅니다.

**출력에서 볼 것**: 정답성 판정이 답 내용과 맞는지 읽어 봅니다. 답이 「발췌에 없다」고 하면 정답 문서의 어느 절이 발췌에서 빠졌는지는 예제 2-6 (3) 에서 발췌와 절 목록을 열어 확인합니다. 어떤 문항이 나오는지는 실행마다 달라질 수 있습니다.

In [ ]:
# 리랭킹 결과에서 정답성이 '예' 가 아닌 문항을 골라 앞 두 건의 질문·답 앞부분을 출력합니다
wrong = [row for row in ROWS['rerank']
         if row['gold'] not in (None, 'NONE') and row['correct'] != '예']
for row in wrong[:2]:
    print(row['gold'], '| 정답성', row['correct'])
    print('  질문', row['q'][:40])
    # 답 앞 90자를 줄바꿈 없이 한 줄로
    print('  답  ', row['ans'][:90].replace('\n', ' '))

# Claude Code 에 물어보기: wrong 을 정답성 '부분' 과 '아니오' 로 나눠 각각 몇 건인지 집계해 줘

## 예제 2-5. 문서에 없는 질문으로 거절을 확인합니다

**1. 없는 호기를 묻는다**

**이 셀이 하는 일**: 문서에 없는 호기(ETCH-07)를 묻는 질문으로 검색기 3종마다 `answer` 를 호출하고, 발췌 문서번호(앞 3개)와 답의 앞부분을 출력합니다.

**출력에서 볼 것**: 발췌에 비슷한 ETCH-04 문서(QIR-2026-0712, SOP-ETCH-007)가 들어오는데도 검색기 3종 모두 답이 거절 문장으로 시작하는지 확인합니다. 검색은 없는 호기를 물어도 가장 비슷한 문서를 돌려줍니다. 거절을 만든 규칙은 예제 2-5 (2) 에서 확인합니다.

In [ ]:
# 문서에 없는 호기(ETCH-07) 질문을 세 검색기 모드에 넣어 발췌 문서번호와 답 앞부분을 출력합니다
q = 'ETCH-07 호기 ESC 온도 기준은?'
for mode in ['vector', 'hybrid', 'rerank']:
    ans, hits = answer(q, mode)
    # 발췌 문서번호를 중복 없이 정렬해 앞 3개만 출력
    doc_ids = sorted({h.doc_id for h in hits})[:3]
    print(f'{mode:7s}', doc_ids, '→', ans[:40].replace('\n', ' '))

# Claude Code 에 물어보기: search(q) 로 ETCH-07 질문의 상위 점수를 출력해 ETCH-04 질문과 비교해 줘

**2. 거절을 만든 프롬프트 규칙을 확인한다**

(심화 · 수업에서 넘어갈 수 있음) 넘어가도 뒤의 기본 셀은 이 셀이 만든 이름을 쓰지 않아 그대로 실행됩니다.

**이 셀이 하는 일**: 답변 프롬프트 `TEMPLATE` 를 줄마다 훑어 「근거를 찾지」가 들어간 줄만 출력합니다.

**출력에서 볼 것**: 3일차에 넣은 근거 없음 규칙 줄(「발췌에 없으면 … 라고만 답한다」) 한 줄이 나옵니다. 예제 2-5 (1) 의 거절은 이 규칙이 만듭니다. 예제 2-4 의 `refused` 도 답이 이 문장으로 시작하는지로 판정합니다.

In [ ]:
# 답변 프롬프트 TEMPLATE 에서 근거 없음 규칙 줄만 골라 출력합니다
for line in TEMPLATE.splitlines():
    if '근거를 찾지' in line:
        print(line)

# Claude Code 에 물어보기: 거절 문장을 바꾸면 eval_one 의 refused 판정이 왜 깨지는지 짚어 줘

**3. 있는 질문에서 거절한 건수도 같이 집계한다**

**이 셀이 하는 일**: 예제 2-4 의 `ROWS` 에서 검색기마다 정답 문서가 있는 질문과 없는 질문(gold 가 'NONE')을 나누어 거절 건수를 집계합니다. 정답 문서를 정하지 않은 1문항(gold 가 None)은 양쪽에서 모두 빠집니다.

**출력에서 볼 것**: 없는 질문 1개는 거절했는지, 있는 질문 31개에서는 몇 건을 거절했는지 봅니다. 거절만 늘면 있는 내용도 없다고 답하므로 두 숫자를 같이 봅니다. 거절 건수는 LLM 답에 따라 실행마다 달라질 수 있습니다. 있는 질문의 거절에는 발췌에 정답 문서가 없어서 거절한 건도 섞일 수 있습니다. 발췌에 정답이 있었는지(`retrieved`)와 정답성을 함께 보는 일은 예제 2-6 (1) 에서 합니다.

In [ ]:
# 검색기마다 정답 문서가 있는 질문과 없는 질문으로 나눠 거절 건수를 집계합니다
for mode, rows in ROWS.items():
    with_gold = [r for r in rows if r['gold'] not in (None, 'NONE')]
    no_gold = [r for r in rows if r['gold'] == 'NONE']
    # True 를 1 로 더해 거절 건수를 집계합니다
    refused_with_gold = sum(r['refused'] for r in with_gold)
    refused_no_gold = sum(r['refused'] for r in no_gold)
    print(f'{mode:7s} 있는 질문 {len(with_gold)}개 중 거절 {refused_with_gold}건 ·',
          f'없는 질문 {len(no_gold)}개 중 거절 {refused_no_gold}건')

# Claude Code 에 물어보기: 있는 질문의 거절 건을 retrieved 참·거짓으로 나눠 집계해 줘

## 예제 2-6. 병목이 검색인가 생성인가

**1. 실패 유형 네 칸으로 문항을 나눈다**: 정답 문서가 발췌에 들어왔는가 × 답이 맞았는가

**이 셀이 하는 일**: `four_box` 는 정답 문서가 있는 문항을 「정답 문서가 발췌에 들어왔는가(`retrieved`)」와 「정답성이 '예' 인가('부분' · '아니오' 는 틀림으로 판정합니다)」 두 기준으로 네 칸에 나눠 집계합니다.

**출력에서 볼 것**: 못 찾은 칸(못 찾고 틀림·못 찾고 맞음)이 거의 비어 있고 대부분이 「찾고 틀림」에 몰렸는지 봅니다. 그렇다면 정답 문서는 찾고 있으므로, 검색기 교체보다 생성 단계(프롬프트 · 발췌 개수 · 청크 단위)를 먼저 봅니다. `retrieved` 는 문서 단위라 정답 문서의 다른 절만 들어온 경우도 「찾음」으로 판정합니다. 이런 문항은 예제 2-6 (3) 에서 엽니다. 정답성은 LLM 판정이라 칸별 숫자는 실행마다 달라질 수 있습니다.

In [ ]:
def four_box(rows):
    """채점 행 목록을 받아 발췌 적중 × 정답 여부 네 칸의 건수를 돌려줍니다."""
    scored_rows = [r for r in rows if r['gold'] not in (None, 'NONE')]
    # 문항마다 (정답 문서를 찾았는지, 답이 맞았는지) 짝을 만듭니다
    pairs = [(r['retrieved'], r['correct'] == '예') for r in scored_rows]
    return {'찾고 맞음': sum(found and right for found, right in pairs),
            '찾고 틀림': sum(found and not right for found, right in pairs),
            '못 찾고 틀림': sum(not found and not right for found, right in pairs),
            '못 찾고 맞음': sum(not found and right for found, right in pairs)}

for mode, rows in ROWS.items():
    print(f'{mode:7s}', four_box(rows))

# Claude Code 에 물어보기: four_box 결과를 2×2 표 모양으로 출력하는 코드 짜 줘

**2. 칸마다 다음에 고칠 곳을 정한다**

**이 셀이 하는 일**: 칸마다 다음에 고칠 곳(`NEXT`)을 붙이고, 리랭킹 결과의 칸을 건수가 많은 순으로 출력합니다. 「찾고 맞음」 칸은 고칠 곳이 없어 `NEXT` 에 넣지 않았습니다.

**출력에서 볼 것**: 맨 위 줄이 가장 큰 칸이고, 화살표 뒤가 그 칸의 다음에 고칠 곳입니다. 「찾고 틀림」이 가장 크면 검색기를 더 바꾸기보다 생성 단계(프롬프트·발췌 개수·청크)를 먼저 고칩니다.

In [ ]:
# 칸마다 다음에 고칠 곳을 붙이고 리랭킹 결과를 건수가 많은 칸부터 출력합니다
NEXT = {'찾고 틀림': '생성 단계 · 프롬프트 · 발췌 개수 · 청크',
        '못 찾고 틀림': '검색 단계 · 검색기 · 색인 · 질문',
        '못 찾고 맞음': '평가셋 · 정답 라벨 점검'}
box = four_box(ROWS['rerank'])
# 건수가 많은 칸부터 정렬합니다
for box_name in sorted(NEXT, key=lambda x: -box[x]):
    print(f'{box_name:7s} {box[box_name]:2d}건 → {NEXT[box_name]}')

# Claude Code 에 물어보기: NEXT 에 '찾고 맞음' 칸도 넣어 네 칸이 모두 출력되게 고쳐 줘

**3. 찾고 틀린 문항 하나를 연다**: 발췌에 정답 문서가 있었는데 왜 틀렸는지

(심화 · 수업에서 넘어갈 수 있음) 넘어가도 뒤의 기본 셀은 이 셀이 만든 이름을 쓰지 않아 그대로 실행됩니다.

**이 셀이 하는 일**: 리랭킹 결과에서 정답 문서가 SPEC-027 인 문항을 가져와 질문·판정·답 앞부분, 받은 발췌 4건, 정답 문서의 절 목록을 출력합니다. `DOCT` 는 환경 준비 (3) 에서 만든 문서번호 → 청크 목록 사전입니다.

**출력에서 볼 것**: 「받은 발췌」 아래 4줄이 모두 서로 다른 공정스펙의 「관리 항목」 절인지 봅니다. 정답 문서의 절 목록 첫 항목인 문서 머리(레시피·호기가 적힌 절)가 발췌에 없으면, 어느 표가 질문의 레시피 RCP-CMP-191 의 것인지 가릴 근거가 없습니다. 이런 문항이면 다음에 고칠 곳 하나로 청크 단위(문서 머리의 레시피 · 호기를 절 청크에도 붙이기)를 적어 둡니다.

In [ ]:
# 찾고 틀린 문항(SPEC-027) 하나의 질문·판정·답 앞부분·발췌·정답 문서 절 목록을 출력합니다
row = next(x for x in ROWS['rerank'] if x['gold'] == 'SPEC-027')
print('질문', row['q'][:46])
print('정답성', row['correct'], '| 답', row['ans'][:30])
print('받은 발췌')
for excerpt in row['excerpt']:
    print('  ', excerpt)
# 정답 문서 청크들의 절 제목(앞 14자)
print('정답 문서의 절', [c.section[:14] for c in DOCT['SPEC-027']])

# Claude Code 에 물어보기: SPEC-027 의 문서 머리 청크가 rerank_rank 에서 몇 위인지 찾아 줘

## 파트 3 · 예제 3-1 비교용 · RAG 로 같은 질문

파트 3 의 예제 3-1 에서는 Claude Code 로 ETCH ESC 문서 21건을 위키로 정리하고, 위키에 두 질문을 물었습니다(`day04-wiki/prompts/2_위키질문.txt`). 이 절은 같은 두 질문을 오늘 만든 RAG 에 그대로 넣어, 위키의 답과 나란히 놓고 볼 RAG 쪽 답을 만듭니다.

**이 셀이 하는 일**: 예제 2-3 (1) 셀의 `answer(q, mode, k)` 로 두 질문을 하이브리드 · 리랭킹 검색기에 넣고(발췌 4건), 질문 번호 · 검색기 · 걸린 시간 줄 아래에 발췌로 들어온 문서와 절, 답 전체를 출력합니다. 색인은 환경 준비 (3) 의 문서 252건 · 754청크 그대로입니다. 위키는 ETCH ESC 문서 21건으로 만들었고 RAG 는 252건 전체 색인에서 발췌 4건으로 답하므로, 비교할 때 재료 범위와 발췌 건수가 다른 점을 감안합니다.
- 위에서부터 예제 2-3 (1) 셀까지 실행한 뒤 이 셀을 실행합니다. `answer` 가 없다는 NameError 가 나면 환경 준비 (1) ~ (4) 와 예제 1-1 (2) · 1-1 (4) · 1-2 (1) · 1-3 (1) · 1-3 (3) · 1-4 (1) · 1-4 (2) · 2-3 (1) 셀을 차례로 실행합니다.
- 질문 두 개에 검색기 둘이라 LLM 호출은 네 번입니다.

**출력에서 볼 것**: 질문 1 에서 「발췌」 줄에 ETCH-04 온도 드리프트 보고서가 들어오는지, 질문 2 에서 척킹 불량 건수와 공통 원인을 RAG 가 어디까지 답하는지 봅니다. 발췌 4건으로 여러 문서를 모아야 하는 질문에 답이 모이는지, 위키의 답과 비교합니다.

In [ ]:
# 예제 3-1 비교용 · 위키에 물은 두 질문을 RAG 에 그대로 넣습니다 (예제 2-3 (1) 의 answer 를 씁니다)
QS = ['ETCH-04 의 다음 PM 을 정기 일정보다 앞당겨야 합니까? '
      '판단 근거와 아직 확인이 필요한 것을 알려 주세요.',
      'ETCH 계열 척킹 불량은 지금까지 몇 건이었고, 공통 원인과 조치는 무엇입니까?']
for question_no, q in enumerate(QS, 1):
    for mode in ['hybrid', 'rerank']:
        started = time.perf_counter()
        ans, hits = answer(q, mode, 4)
        print(f'===== 질문 {question_no} · {mode} · {time.perf_counter() - started:.1f}초')
        print('발췌', [h.doc_id + ' · ' + h.section[:14] for h in hits])
        print(ans)
        print()

# Claude Code 에 물어보기: '척킹' 이 든 청크가 색인에 몇 개인지 집계해 발췌 4건과 비교해 줘